![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 10 -- Lab 1: Six Models One Table

The admissions task one more time, now with the whole toolbox: k-nearest neighbours, a support vector machine, a decision tree, a random forest and gradient boosting, next to the logistic regression you already have. Every model is scored the same way, on the same validation rows, by the organisers' metric (ROC-AUC). You will see each model's dial move the score, read a forest's importances, tune gradient boosting with a grid search, and hand in the best model's probabilities.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** `train.csv` (6,885 students) and `test.csv` (765) from the HAIO 2025 repository, read from GitHub. The setup cell renames the columns and applies Day 08's preparation, so you start from `X_train, X_val, y_train, y_val` and the test table `X_test`.

---
# Setup

In [ ]:
# pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q pandas matplotlib scikit-learn

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
BASE = "https://raw.githubusercontent.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad/main/2025/nyari-online/adatok/"
EN = {"Életkor": "age", "Nem": "sex", "Osztályzat_9": "grade_9", "Osztályzat_10": "grade_10", "Osztályzat_11": "grade_11",
      "Osztályzat_12": "grade_12", "Történelem": "history", "Matematika": "math", "Magyar Nyelv és Irodalom": "hungarian",
      "Informatika": "informatics", "Biológia": "biology", "Fizika": "physics", "Angol": "english", "Német": "german",
      "Informatika_emelt": "informatics_adv", "Biológia_emelt": "biology_adv", "Fizika_emelt": "physics_adv",
      "Angol_emelt": "english_adv", "Német_emelt": "german_adv", "Matematika_emelt": "math_adv", "Történelem_emelt": "history_adv",
      "Magyar Nyelv és Irodalom_emelt": "hungarian_adv", "Szülői Végzettség": "parent_education",
      "Középiskola Presztízse": "school_prestige", "Extrakurrikuláris Tevékenységek": "extracurricular",
      "Tanulási Szokások": "study_habits", "Munkatapasztalat": "work_experience", "Ajánlások Száma": "recommendations",
      "Versenyeken Való Részvétel": "competitions", "Vármegye": "county", "Felvételi Eredmény": "admitted", "ID": "ID"}
train = pd.read_csv(BASE + "train.csv").rename(columns=EN)
test = pd.read_csv(BASE + "test.csv").rename(columns=EN)

In [ ]:
def prepare(d):
    """Day 08 Lab 3 in one function: -1 becomes a flag plus a zero, the county becomes one-hot columns."""
    d = d.copy()
    for c in ["informatics", "biology", "physics", "english", "german"]:
        d[c + "_taken"] = (d[c] != -1).astype(int)
        d[c] = d[c].replace(-1, 0)
        d[c + "_adv"] = d[c + "_adv"].replace(-1, 0)
    feats = [c for c in d.columns if c not in ("county", "admitted", "ID")]
    return pd.concat([d[feats], pd.get_dummies(d["county"], prefix="county", dtype=int)], axis=1)

X = prepare(train)
y = train["admitted"].astype(int)
X_test = prepare(test).reindex(columns=X.columns, fill_value=0)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape)

---
# Part 1 -- One scoring function for every model

The *No Free Lunch* slide: models are compared on the task's metric, on rows they never saw. Write the scorer once and reuse it.

## Task 1: The scorer

Write `evaluate(model)` that fits `model` on `X_train, y_train`, computes the ROC-AUC of `predict_proba(X_val)[:, 1]` and the accuracy of `predict(X_val)`, prints both with 3 decimals, and returns the ROC-AUC. Test it on `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`: expected ROC-AUC 0.954, accuracy 0.871. Store the result in `results = {"logistic": ...}`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['logistic'] - 0.954) < 0.005, 'evaluate: fit on train, ROC-AUC of predict_proba[:, 1] on X_val'
print('Task 1 passed')

---
# Part 2 -- k-nearest neighbours

The *Scaling Is Not Optional* and *Choosing k* slides.

## Task 2: Scaled against unscaled

Evaluate `KNeighborsClassifier(15)` twice: once on the raw columns, once inside `make_pipeline(StandardScaler(), ...)`. Expected ROC-AUC 0.697 unscaled (accuracy 0.699, below the 0.705 majority baseline) and 0.850 scaled.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['knn'] - 0.850) < 0.005, 'knn: StandardScaler then KNeighborsClassifier(15), evaluated with the scorer'
print('Task 2 passed')

## Task 3: The dial k

For `k` in `[1, 5, 15, 51]` evaluate the scaled KNN and note the ROC-AUC. Expected: 0.681, 0.820, 0.850, 0.851. Why is `k = 1` the worst although it scores 100% on the training rows?

In [ ]:
# Your code here


---
# Part 3 -- Support vector machine

The *Widest Street* and *Kernels* slides. `probability=True` makes `predict_proba` available; it costs a few seconds.

## Task 4: An RBF support vector machine

Evaluate `make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, probability=True, random_state=0))` and store it as `results["svm"]`. Expected ROC-AUC about 0.942. Time the call with `time.time()` before and after and print the seconds.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['svm'] - 0.942) < 0.01, 'svm: scaled SVC with probability=True'
print('Task 4 passed')

---
# Part 4 -- Decision tree

The *Twenty Questions* and *max_depth* slides. No scaler here.

## Task 5: The dial max_depth

For `depth` in `[2, 4, 6, 10, None]` evaluate `DecisionTreeClassifier(max_depth=depth, random_state=0)`. Expected ROC-AUC 0.724, 0.874, 0.899, 0.821, 0.758. Store the depth-6 score as `results["tree"]`. Which depth is best, and what happens past it?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['tree'] - 0.899) < 0.005, 'tree: DecisionTreeClassifier(max_depth=6, random_state=0)'
print('Task 5 passed')

## Task 6: Draw the top of the tree

Fit `DecisionTreeClassifier(max_depth=2, random_state=0)` and draw it with `plot_tree(tree, feature_names=list(X.columns), class_names=["no", "yes"], filled=True, fontsize=8)` in a figure of size (12, 5). Read the root question and write it in a comment.

In [ ]:
# Your code here


---
# Part 5 -- Random forest and gradient boosting

The *Ensembles* slides.

## Task 7: A forest of 300 trees

Evaluate `RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)` as `results["forest"]` (expected ROC-AUC about 0.938) and keep the fitted model in `forest`. Compare with the single depth-6 tree.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['forest'] - 0.938) < 0.01, 'forest: 300 trees, random_state=0'
print('Task 7 passed')

## Task 8: Feature importances

Put `forest.feature_importances_` in a `pd.Series` indexed by `X.columns`, sort it, and draw a horizontal bar chart of the ten largest with a title. Expected top: `math_adv` (0.107), `math` (0.079), `county_Budapest` (0.075).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert importance.idxmax() == 'math_adv', 'importance: pd.Series(forest.feature_importances_, index=X.columns)'
print('Task 8 passed')

## Task 9: Gradient boosting

Evaluate `GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=0)` as `results["boosting"]`. Expected ROC-AUC about 0.946.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['boosting'] - 0.946) < 0.01, 'boosting: 200 trees of depth 3'
print('Task 9 passed')

---
# Part 6 -- Tune, compare, submit

The *GridSearchCV* and *No Free Lunch* slides.

## Task 10: Grid search on gradient boosting

Run `GridSearchCV(GradientBoostingClassifier(random_state=0), {"n_estimators": [100, 300], "max_depth": [2, 3]}, cv=3, scoring="roc_auc")` on `X_train, y_train` (about 10 seconds). Print `best_params_` and `best_score_`, then evaluate `grid.best_estimator_` on the validation rows. Expected best: 300 trees of depth 2, CV 0.944, validation about 0.947. Store it as `results["boosting_tuned"]`.

Syntax hint (the shape of the call, not the answer):

```python
grid = GridSearchCV(estimator, {"dial": [value1, value2]}, cv=3, scoring="roc_auc")
grid.fit(X_train, y_train)     # grid.best_params_, grid.best_score_, grid.best_estimator_
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert grid.best_params_ == {'max_depth': 2, 'n_estimators': 300}, 'grid: the four combinations scored by 3-fold ROC-AUC on the training rows'
print('Task 10 passed')

## Task 11: The comparison table

Turn `results` into a `pd.Series` sorted from best to worst and print it. Which model wins this table? Was it the most complex one?

In [ ]:
# Your code here


## Task 12: Submit the best model's probabilities

Take the best model of the table, refit it on all of `X, y`, and save `pd.DataFrame({"ID": test["ID"], "admitted": probabilities})` (probabilities from `predict_proba(X_test)[:, 1]`) to `submission.csv` with `index=False`. Print the shape.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pd.read_csv('submission.csv').shape == (765, 2), 'submission.csv: ID and admitted (probabilities), 765 rows'
print('Task 12 passed')

---
## Task 13: Reflect

KNN and the SVM needed a scaler and the trees did not: explain why from how each one makes a prediction. The single tree peaked at depth 6 and the forest of unlimited-depth trees still scored higher: what does the forest do that the tree cannot? Logistic regression beat five fancier models here. What would you try next on this table, and how would you know if it worked?

*Your answers here*

---
## Conclusion

- **One scorer**: every model was judged by the same function, on the same validation rows, by the organisers' metric.
- **Dials**: k, C and the kernel, max_depth, n_estimators; each one moved the score, and each one has an overfitting side.
- **Ensembles and importances**: the forest and the boosting beat the single tree, and the forest told you which columns it used.
- **Tuning and choosing**: grid search picked the boosting dials by cross-validation, and the final table showed that the simplest model won this task.

---

Made By **Sattam Altwaim**